In [ ]:
import webdataset as wds
from torchvision import transforms
from torch.utils.data import default_collate

def make_dataloader_train(trainset_url, cache_dir='./_cache', batch_size=32):
    """Create a DataLoader for training on the ImageNet dataset using WebDataset."""

    # This is the basic WebDataset definition: it starts with a URL and add shuffling,
    # decoding, and augmentation. Note `resampled=True`; this is essential for
    # distributed training to work correctly.
    transform = transforms.Compose(
            [
                transforms.ToTensor(),
            ]
        )
    def make_sample(sample):
        return (transform(sample["png"]), )

    trainset = wds.WebDataset(
        trainset_url,
        resampled=True,
        shardshuffle=True,
        cache_dir=cache_dir,
        nodesplitter=wds.split_by_node,
    )
    trainset = trainset.shuffle(1000).decode("pil").map(make_sample)

    # For IterableDataset objects, the batching needs to happen in the dataset.
    trainset = trainset.batched(batch_size)

    trainloader = wds.WebLoader(trainset, batch_size=None, num_workers=4)

    # We unbatch, shuffle, and rebatch to mix samples from different workers.
    trainloader = trainloader.unbatched().shuffle(1000).batched(batch_size)

    # A resampled dataset is infinite size, but we can recreate a fixed epoch length.
    trainloader = trainloader.with_epoch(1282 * 100 // 64)

    return trainloader

trainloader = make_dataloader_train("/home/btang5/work/2025/data/Imagenet64_process_script/data/imagenet_tars/imagenet64-{0001..0100}.tar")

for d in trainloader:
    print(d[0].shape)


: 